In [3]:
# scripts/gen_config_docs.py
import sys
from pathlib import Path

# Path to the project root: /mnt/home/.../scripts/boettiger-livecell
PROJECT_ROOT = Path("/mnt/home/tudomlumleart/ceph/04_MitoticChromosome/scripts/boettiger-livecell")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    
from typing import Type
from pydantic import BaseModel
from simulation.sim_config import SimConfig, SlurmCfg  

In [ ]:

def model_to_markdown(model: Type[BaseModel], title: str) -> str:
    lines = []
    lines.append(f"### {title}\n")
    lines.append("| Name | Type | Default | Constraints | Description |")
    lines.append("| ---- | ---- | ------- | ----------- | ----------- |")

    for name, field in model.model_fields.items():
        # skip private attrs like _num_cohesin_types
        if name.startswith("_"):
            continue

        # Type
        type_str = getattr(field.annotation, "__name__", str(field.annotation))

        # Default
        default = field.default if field.default is not None else "null"

        # Constraints (for the README, we just stringify)
        constraints = []
        if field.metadata:
            for meta in field.metadata:
                for k, v in vars(meta).items():
                    # crude filter for common constraint keys
                    if k in {"ge", "gt", "le", "lt", "max_length", "min_length"} and v is not None:
                        constraints.append(f"{k}={v}")
        constraint_str = ", ".join(constraints) if constraints else "–"

        # Description
        desc = field.description or ""

        lines.append(f"| `{name}` | `{type_str}` | `{default}` | `{constraint_str}` | {desc} |")

    return "\n".join(lines)


if __name__ == "__main__":
    print("# Configuration\n")
    print(model_to_markdown(SlurmCfg, "Slurm configuration (`SlurmCfg`)"))
    print()
    print(model_to_markdown(SimConfig, "Simulation configuration (`SimConfig`)"))
